# **Notebook 1d: Splitting the manually labeled data for model training and evaluation**

- **Test** (~25%)
- **Validation** (~15%)
- **Train** (~60%)

## 1. Importing the packages

In [ ]:
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

## 2. Loading the data

In [ ]:
labeled_data = pd.read_excel("labeled_data.xlsx")
print(f"Total sentences: {len(labeled_data)}")
print(f"Total blurbs:    {labeled_data['blurb_id'].nunique()}")
labeled_data.head()

#------------------------------------------------------------------------------------------
# Output deactivated for data privacy concerns
# Total sentences: 1887
# Total blurbs:    200

## 3. Checking spread of labels across blurbs

In [ ]:
LABEL_COL = "unnecessary"
labels_per_blurb = labeled_data.groupby("blurb_id")[LABEL_COL].agg(["mean", "nunique", "size"])

print(f"Single-class blurbs (only 0 or only 1): {(labels_per_blurb['nunique'] == 1).sum()}")
print(f"Mixed-class blurbs                    : {(labels_per_blurb['nunique'] > 1).sum()}")
print()
print("Distribution of blurb purity (mean label):")
print(labels_per_blurb['mean'].describe())

Single-class blurbs (only 0 or only 1): 69
Mixed-class blurbs                    : 131

Distribution of blurb purity (mean label):
count    200.000000
mean       0.228765
std        0.222602
min        0.000000
25%        0.000000
50%        0.211111
75%        0.375000
max        1.000000
Name: mean, dtype: float64


In [ ]:
labeled_data['unnecessary'].value_counts(normalize=True)

,proportion
unnecessary,
0,0.731849
1,0.268151


## 4. Settings

In [ ]:
# Proportion of blurbs reserved for the final hold-out test set.
TEST_PROPORTION = 0.25
# Proportion of total reserved for validation (= 20% of remaining 75%).
VAL_PROPORTION = 0.15
# Random seed for reproducibility of the split.
RANDOM_STATE = 222

## 5. Preparing data for splitting

In [ ]:
# Resetting index so row positions match the integer indices used below.
df = labeled_data.reset_index(drop=True).copy()

# Arrays used by the splitter:
row_indices = df.index.values
labels = df[LABEL_COL].values
blurb_ids = df["blurb_id"].values

## 6. Splitting into Test and Dev (Train+Val)

Step 1: Holding out the test set (~25% of blurbs).

In [ ]:
# 4 splits chosen so one split equals the test proporion of 25%.
test_n_splits = max(2, round(1 / TEST_PROPORTION))
test_splitter = StratifiedGroupKFold(
    n_splits=test_n_splits,
    shuffle=True,
    random_state=RANDOM_STATE,)

# Taking the first split: dev_indices = the rest, test_indices = the held-out fold.
dev_indices, test_indices = next(test_splitter.split(row_indices, labels, blurb_ids))

# Showing statistics
test_blurbs = set(blurb_ids[test_indices])
dev_blurbs = set(blurb_ids[dev_indices])

print(f"Dev set  : {len(dev_indices):>4} sentences | {len(dev_blurbs):>3} blurbs | {labels[dev_indices].mean():.1%} positive")
print(f"Test set : {len(test_indices):>4} sentences | {len(test_blurbs):>3} blurbs | {labels[test_indices].mean():.1%} positive")

Dev set  : 1406 sentences | 148 blurbs | 28.1% positive
Test set :  481 sentences |  52 blurbs | 23.1% positive


## 7. Splitting Dev into Train and Validation

Step 2: From the remaining 75% (dev set), hold out the validation set.

In [ ]:
# Computing proportion of validation INSIDE dev set.
val_proportion_of_dev = VAL_PROPORTION / (1 - TEST_PROPORTION)   # 0.15 / 0.75 = 0.20
val_n_splits = max(2, round(1 / val_proportion_of_dev))

# Subsetting arrays to dev only (the splitter only operates on this subset).
dev_labels = labels[dev_indices]
dev_blurb_ids = blurb_ids[dev_indices]

val_splitter = StratifiedGroupKFold(
    n_splits=val_n_splits,
    shuffle=True,
    random_state=RANDOM_STATE,)

# Splitter returns indices RELATIVE to dev_indices; map back to global indices in df.
train_rel, val_rel = next(val_splitter.split(dev_indices, dev_labels, dev_blurb_ids))
train_indices = dev_indices[train_rel]
val_indices = dev_indices[val_rel]

# Showing statistics
train_blurbs = set(blurb_ids[train_indices])
val_blurbs = set(blurb_ids[val_indices])

print(f"Train set : {len(train_indices):>4} sentences | {len(train_blurbs):>3} blurbs | {labels[train_indices].mean():.1%} positive")
print(f"Val set   : {len(val_indices):>4} sentences | {len(val_blurbs):>3} blurbs | {labels[val_indices].mean():.1%} positive")
print(f"Test set  : {len(test_indices):>4} sentences | {len(test_blurbs):>3} blurbs | {labels[test_indices].mean():.1%} positive")

Train set : 1104 sentences | 119 blurbs | 28.5% positive
Val set   :  302 sentences |  29 blurbs | 26.5% positive
Test set  :  481 sentences |  52 blurbs | 23.1% positive


## 8. Sanity check (no leakage, and balanced splits)

In [ ]:
# Check 1: No blurb appears in multiple splits
assert len(train_blurbs & val_blurbs) == 0, "Leakage: blurbs shared between train and val!"
assert len(train_blurbs & test_blurbs) == 0, "Leakage: blurbs shared between train and test!"
assert len(val_blurbs & test_blurbs) == 0, "Leakage: blurbs shared between val and test!"
print("No blurb leakage between splits")

# Check 2: All sentences are accounted for
total_assigned = len(train_indices) + len(val_indices) + len(test_indices)
assert total_assigned == len(df), f"Sentence count mismatch: {total_assigned} vs {len(df)}"
print(f"All {len(df)} sentences assigned to a split")

# Check 3: Label balance is similar across splits
print("\nLabel distribution per split:")
print(f"  Train: {labels[train_indices].mean():.1%} positive")
print(f"  Val:   {labels[val_indices].mean():.1%} positive")
print(f"  Test:  {labels[test_indices].mean():.1%} positive")

No blurb leakage between splits
All 1887 sentences assigned to a split

Label distribution per split:
  Train: 28.5% positive
  Val:   26.5% positive
  Test:  23.1% positive


## 9. Saving splits to CSV

In [ ]:
train_df = df.iloc[train_indices]
val_df = df.iloc[val_indices]
test_df = df.iloc[test_indices]

train_df.to_csv("labeled_train.csv", index=False)
val_df.to_csv("labeled_val.csv", index=False)
test_df.to_csv("labeled_test.csv", index=False)

print("Saved:")
print(f"  labeled_train.csv  ({len(train_df)} sentences, {train_df['blurb_id'].nunique()} blurbs)")
print(f"  labeled_val.csv    ({len(val_df)} sentences, {val_df['blurb_id'].nunique()} blurbs)")
print(f"  labeled_test.csv   ({len(test_df)} sentences, {test_df['blurb_id'].nunique()} blurbs)")

Saved:
  labeled_train.csv  (1104 sentences, 119 blurbs)
  labeled_val.csv    (302 sentences, 29 blurbs)
  labeled_test.csv   (481 sentences, 52 blurbs)
